In this work I used data and project @nasimetemadi

## Fraud Detection in Financial Transactions
### Problem
Identifying fraudulent transactions is a crucial yet complex task within contemporary financial systems. Traditional supervised machine learning methods depend on datasets where fraudulent and legitimate transactions are distinctly labeled. However, in practical applications, such labeled data is often scarce, significantly imbalanced, or not available at all. This situation calls for unsupervised techniques capable of detecting anomalies without pre-existing label information. The primary challenge lies in accurately identifying fraudulent activities while minimizing false positives and ensuring the model's scalability for large datasets. Models typically underperform on datasets with severe class imbalance, where one class represents a tiny fraction of the data.
### Solution
To tackle this issue, we employed the KMeans model, foregoing the use of labeled data for training purposes. Labeled data was utilized solely for preprocessing. Our approach involved:
- Selecting features that distinctly differed between the two classes,
- Reducing the volume of class 0 data to achieve a more balanced dataset,
- Training the model on this balanced dataset without supervision,
- Testing the model on the complete, highly imbalanced dataset, achieving a precision of 0.73 and a recall of 0.75 for class 1 (fraudulent transactions) recognition.

This method aligns with practices highlighted in various studies, such as those discussing the benefits of unsupervised learning in fraud detection source, and emphasizes the importance of preprocessing to balance datasets for improved model performance source.

In [598]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

In [599]:
data = pd.read_csv('creditcard.csv')
print(pd.Series({"Memory usage": "{:.2f} MB".format(data.memory_usage().sum()/(1024*1024)),
                 "Dataset shape": "{}".format(data.shape)}).to_string())
data.head()

Memory usage         67.36 MB
Dataset shape    (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [600]:
#data.info()

In [601]:
#data.describe().style.set_sticky(axis="index")

In [602]:
Class = data['Class']
data_fraud = data[data['Class'] == 1]
data_auth = data[data['Class'] == 0]
data_auth.shape, data_fraud.shape

((284315, 31), (492, 31))

Let's split the data so that the data for training the model is more balanced across classes. The model learns better this way.

In [603]:
data_auth_train, data_auth_test = train_test_split(data_auth, test_size=0.95, random_state=42)
data_auth_train.shape, data_auth_test.shape

((14215, 31), (270100, 31))

In [604]:
data_fraud_train, data_fraud_test = train_test_split(data_fraud, test_size=0.4, random_state=42)
data_fraud_train.shape, data_fraud_test.shape

((295, 31), (197, 31))

In [605]:
features = data.drop(columns='Class', errors='ignore' ).columns

fig = plt.figure(figsize=(15, 40))

for i, column in enumerate(features):    
    plt.subplot(12, 3, i+1)
    sns.histplot(data=data_auth_train, x=column, kde=True, stat='density', color='green')
    sns.histplot(data=data_fraud, x=column, kde=True, stat='density', color='red')
           
plt.show()

We will select only those features that are clearly different in these two classes.

In [606]:
selected_features = ['V2','V3','V4','V7','V9','V10','V11','V12','V14','V16','V17']

In [607]:
X = pd.concat([data_auth_train, data_fraud_train], axis=0)
X = X[selected_features]
X.shape

(14510, 11)

In [608]:
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
km = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)
labels = km.fit_predict(X_sc)

In [609]:
def show_metrics(data, data_fraud, labels):
    data['Class'] = Class
    data['labels'] = labels
    cluster_counts = data['labels'].value_counts()
    cluster = cluster_counts.idxmin()
    cluster_data = data[data['labels'] == cluster]
    precision = len(cluster_data.index.intersection(data_fraud.index)) / len(cluster_data)
    recal = len(cluster_data.index.intersection(data_fraud.index)) / len(data_fraud)
    print(f'Metrics for class 1 (fraudulent transactions):')
    print(f'precision: {round(precision, 2)}')
    print(f'recal: {round(recal, 2)}')

In [610]:
show_metrics(X, data_fraud_train, labels)

Metrics for class 1 (fraudulent transactions):
precision: 0.97
recal: 0.46


### Let's test the model on complete and highly unbalanced data that it has never seen.

In [611]:
data_test = pd.concat([data_auth_test, data_fraud_test], axis=0)
data_test = data_test[selected_features]
data_test.shape

(270297, 11)

In [612]:
labels = km.predict(data_test)

In [613]:
show_metrics(data_test, data_fraud_test, labels)

Metrics for class 1 (fraudulent transactions):
precision: 0.73
recal: 0.75
